# Pre-extract & Cache Image Features (ViT-L/14 CLIP)

## Mô tả task

Trong pipeline BLIP-2, mỗi ảnh phải qua ViT-L/14 để trích xuất đặc trưng.  
Nếu làm điều đó trong mỗi training step thì sẽ lãng phí rất nhiều GPU vì:
- ViT-L/14 bị frozen (không cập nhật weight) trong toàn bộ quá trình training
- Mỗi ảnh được xem nhiều lần (qua nhiều epoch)

**Giải pháp**: Chạy ViT một lần duy nhất, lưu kết quả vào file `.h5`, training chỉ cần đọc từ cache.

```

## Sơ đồ tổng quan

┌─────────────────── PRE-EXTRACTION (chạy 1 lần) ─────────────────────────┐
│                                                                         │
│  COCO images (train2014/)         COCO images (val2014/)                │
│       82,783 ảnh .jpg                  40,504 ảnh .jpg                  │
│           │                                  │                          │
│           ▼  Batch (B ảnh)                   ▼                          │
│   CLIPImageProcessor                 CLIPImageProcessor                 │
│   resize 224×224 + normalize         resize 224×224 + normalize         │
│           │                                  │                          │
│           ▼                                  ▼                          │
│   ViT-L/14 forward pass              ViT-L/14 forward pass              │
│   last_hidden_state [B, 257, 1024]   last_hidden_state [B, 257, 1024]   │
│           │                                  │                          │
│           ▼  float32 → float16               ▼                          │
│   train_features.h5                  val_features.h5                    │
│   key: str(image_id)                 key: str(image_id)                 │
│   val: float16 [257, 1024]           val: float16 [257, 1024]           │
└─────────────────────────────────────────────────────────────────────────┘

┌──────────────────────── TRAINING (mỗi epoch) ───────────────────────────┐
│                                                                         │
│  train_features.h5  read►  DataLoader  ►  Q-Former  ►  Loss             │
│        (nhanh, trên disk)       [B,257,1024]                            │
└─────────────────────────────────────────────────────────────────────────┘
```

## Đầu vào / Đầu ra chi tiết

### INPUT

| Thành phần | Mô tả |
|---|---|
| Ảnh COCO | `COCO_{split}2014_000000{image_id:012d}.jpg` |
| Annotations JSON | `v2_mscoco_{split}2014_annotations.json` → lọc `image_id` cần thiết |
| Model | `openai/clip-vit-large-patch14` (ViT-L/14 frozen) |

### OUTPUT (cho mỗi image_id)

| Trường | Giá trị |
|---|---|
| Key (HDF5) | `str(image_id)` — ví dụ `"391895"` |
| Value shape | `(257, 1024)` = 257 tokens × 1024 chiều |
| 257 tokens | 1 CLS token + 256 patch tokens (grid 16×16 với patch_size=14 trên ảnh 224px) |
| 1024 dim | Hidden size của ViT-L/14 |
| Dtype | `float16` (giảm 50% dung lượng so với float32) |
| Compression | gzip level 1 |

### Tại sao 257 tokens?

```
Image size = 224 × 224 px
Patch size = 14 × 14 px
Số patches  = (224 ÷ 14) × (224 ÷ 14) = 16 × 16 = 256 patches
+ 1 CLS token (đặc trưng toàn cục)
= 257 tokens
```

### Tiết kiệm bao nhiêu?

| | Không cache | Có cache |
|---|---|---|
| ViT forward/epoch | 82,783 lần | 0 lần |
| GPU dùng cho ViT | ~60-70% | 0% |
| Tốc độ mỗi step | chậm | nhanh 3–5× |

In [ ]:
##  Cell 2: Import & kiểm tra môi trường 
import sys, os, time, warnings, json, struct
warnings.filterwarnings("ignore")

import numpy as np
import torch
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

try:
    import requests
    _REQUESTS_OK = True
except ImportError:
    _REQUESTS_OK = False

try:
    from transformers import CLIPImageProcessor, CLIPVisionModel
    _TRANSFORMERS_OK = True
except ImportError:
    _TRANSFORMERS_OK = False
    print("transformers chưa cài: pip install transformers")

#  Hằng số từ contracts.py 
CLIP_FEATURE_DIM  = 1024   # hidden size ViT-L/14
CLIP_PATCH_TOKENS = 257    # 1 CLS + 256 patch tokens (16×16 grid @ 14px/patch)
IMAGE_SIZE        = 224

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"h5py      : {h5py.__version__}")
print(f"Device    : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM      : {mem_gb:.1f} GB")

Python    : 3.12.13
PyTorch   : 2.10.0+cpu
h5py      : 3.16.0
Device    : cpu


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
##  Cell 3: Cấu hình đường dẫn 
# Chỉnh các đường dẫn này theo môi trường thực tế của bạn.
# Notebook này hỗ trợ chạy "demo mode" nếu không có dữ liệu COCO thật.

CONFIG = {
    #  Thư mục gốc chứa data 
    "data_root"  : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data"),

    #  COCO images 
    "train_img_dir": Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/coco/train2014"),
    "val_img_dir"  : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/coco/val2014"),

    #  VQAv2 annotation (dùng để lọc image_id cần thiết) 
    "train_ann"  : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/vqav2/v2_mscoco_train2014_annotations.json"),
    "val_ann"    : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/vqav2/v2_mscoco_val2014_annotations.json"),

    #  Cache output 
    "cache_dir"      : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/cache"),
    "train_cache_h5" : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/cache/train_features.h5"),
    "val_cache_h5"   : Path("/content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/cache/val_features.h5"),

    #  Model 
    "model_name" : "openai/clip-vit-large-patch14",

    #  Extraction settings 
    "batch_size" : 32,     # giảm xuống nếu OOM (8 hoặc 16)
    "device"     : DEVICE,
}

CONFIG["cache_dir"].mkdir(parents=True, exist_ok=True)

#  Kiểm tra dữ liệu có sẵn không 
HAS_TRAIN_DATA = CONFIG["train_img_dir"].exists() and CONFIG["train_ann"].exists()
HAS_VAL_DATA   = CONFIG["val_img_dir"].exists()   and CONFIG["val_ann"].exists()

print(" Kiểm tra dữ liệu ")
print(f"train images : {'✓' if CONFIG['train_img_dir'].exists() else '✗'}  {CONFIG['train_img_dir']}")
print(f"val images   : {'✓' if CONFIG['val_img_dir'].exists()   else '✗'}  {CONFIG['val_img_dir']}")
print(f"train ann    : {'✓' if CONFIG['train_ann'].exists()     else '✗'}  {CONFIG['train_ann']}")
print(f"val ann      : {'✓' if CONFIG['val_ann'].exists()       else '✗'}  {CONFIG['val_ann']}")
print(f"cache dir    : {CONFIG['cache_dir']}  (đã tạo)")
print()
if HAS_TRAIN_DATA or HAS_VAL_DATA:
    print("Tìm thấy dữ liệu thật → sẽ chạy extraction thật sự")
else:
    print("Không tìm thấy COCO dataset")

─── Kiểm tra dữ liệu ────────────────────────────────────────
train images : ✓  /content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/coco/train2014
val images   : ✓  /content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/coco/val2014
train ann    : ✓  /content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/vqav2/v2_mscoco_train2014_annotations.json
val ann      : ✓  /content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/vqav2/v2_mscoco_val2014_annotations.json
cache dir    : /content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/cache  (đã tạo)

Tìm thấy dữ liệu thật → sẽ chạy extraction thật sự


In [ ]:
##  Cell 4: Load ViT-L/14 model 

print(f"Đang tải model: {CONFIG['model_name']} ...")
t0 = time.time()

processor = CLIPImageProcessor.from_pretrained(CONFIG["model_name"])
vit_model = CLIPVisionModel.from_pretrained(CONFIG["model_name"])
vit_model = vit_model.to(CONFIG["device"]).eval()

# Freeze toàn bộ — không cần gradient vì chỉ inference
for p in vit_model.parameters():
    p.requires_grad_(False)

elapsed = time.time() - t0
total_params = sum(p.numel() for p in vit_model.parameters())

print(f"Model loaded  ({elapsed:.1f}s)")
print()
print(f"{''*50}")
print(f"Model         : {CONFIG['model_name']}")
print(f"Hidden size   : {vit_model.config.hidden_size}")          # 1024
print(f"Num layers    : {vit_model.config.num_hidden_layers}")    # 24
print(f"Num heads     : {vit_model.config.num_attention_heads}")  # 16
print(f"Patch size    : {vit_model.config.patch_size}")           # 14
print(f"Image size    : {vit_model.config.image_size}")           # 224
_grid = vit_model.config.image_size // vit_model.config.patch_size
print(f"Patch grid    : {_grid}×{_grid} = {_grid**2} patches  + 1 CLS = {_grid**2+1} tokens")
print(f"Tổng tham số  : {total_params:,}  (~{total_params/1e6:.0f}M)")
if DEVICE == "cuda":
    mem_used = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM dùng     : {mem_used:.2f} GB")

Đang tải model: openai/clip-vit-large-patch14 ...


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.71G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight   

Model loaded  (19.5s)

──────────────────────────────────────────────────
Model         : openai/clip-vit-large-patch14
Hidden size   : 1024
Num layers    : 24
Num heads     : 16
Patch size    : 14
Image size    : 224
Patch grid    : 16×16 = 256 patches  + 1 CLS = 257 tokens
Tổng tham số  : 303,179,776  (~303M)


In [ ]:
##  Cell 5: Hàm extraction core 
# Đây là hàm trích xuất một batch ảnh → trả về features [B, 257, 1024] float16

def extract_batch_features(
    image_list: list,          # list of PIL.Image
    processor: CLIPImageProcessor,
    model: CLIPVisionModel,
    device: str,
) -> np.ndarray:
    """Forward pass một batch ảnh qua ViT-L/14.

    Args:
        image_list: Danh sách PIL.Image (đã convert RGB).
        processor:  CLIPImageProcessor (resize + normalize).
        model:      CLIPVisionModel đang ở eval mode, frozen.
        device:     "cuda" hoặc "cpu".

    Returns:
        numpy array float16, shape [B, 257, 1024].
        - 257 = 1 CLS + 256 patch tokens
        - 1024 = hidden dim ViT-L/14
    """
    # Bước 1: Tiền xử lý ảnh → tensor [B, 3, 224, 224]
    inputs = processor(images=image_list, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)

    # Bước 2: Forward pass — không cần gradient
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
        # last_hidden_state: [B, 257, 1024]  — toàn bộ tokens kể cả CLS
        features = outputs.last_hidden_state

    # Bước 3: Chuyển về CPU + float16 để tiết kiệm bộ nhớ/disk
    return features.cpu().float().numpy().astype(np.float16)


#  Test với 1 ảnh mẫu 
print("Test extract_batch_features với 1 ảnh ngẫu nhiên ...")
rng   = np.random.default_rng(42)
dummy = Image.fromarray((rng.random((224, 224, 3)) * 255).astype(np.uint8))

t0   = time.time()
feat = extract_batch_features([dummy], processor, vit_model, CONFIG["device"])
dt   = time.time() - t0

print(f"Output shape : {feat.shape}")           # (1, 257, 1024)
print(f"Output dtype : {feat.dtype}")           # float16
print(f"Thời gian    : {dt*1000:.1f} ms (1 ảnh)")
print(f"Size per img : {feat.nbytes / 1024:.1f} KB  (float16)")
print(f"             : {feat.astype(np.float32).nbytes / 1024:.1f} KB  (float32) — gấp đôi")

assert feat.shape == (1, CLIP_PATCH_TOKENS, CLIP_FEATURE_DIM), \
    f"Shape sai! Expect (1,{CLIP_PATCH_TOKENS},{CLIP_FEATURE_DIM}), got {feat.shape}"
print("\n Shape đúng theo contracts.py (CLIP_PATCH_TOKENS=257, CLIP_FEATURE_DIM=1024)")

Test extract_batch_features với 1 ảnh ngẫu nhiên ...
Output shape : (1, 257, 1024)
Output dtype : float16
Thời gian    : 7167.1 ms (1 ảnh)
Size per img : 514.0 KB  (float16)
             : 1028.0 KB  (float32) — gấp đôi

 Shape đúng theo contracts.py (CLIP_PATCH_TOKENS=257, CLIP_FEATURE_DIM=1024)


In [ ]:
##  Cell 6: Hàm ghi cache HDF5 (có resume) 

def extract_and_cache(
    image_ids: list,               # danh sách image_id cần xử lý
    img_dir: Path,                 # thư mục chứa ảnh COCO
    img_prefix: str,               # prefix tên file, ví dụ "COCO_train2014_"
    cache_path: Path,              # đường dẫn file .h5 đầu ra
    processor: CLIPImageProcessor,
    model: CLIPVisionModel,
    device: str,
    batch_size: int = 32,
) -> None:
    """Trích xuất ViT features cho danh sách image_id và lưu vào HDF5.

    Hỗ trợ **resume**: nếu file .h5 đã tồn tại và có một số image_id
    được cache rồi, hàm sẽ bỏ qua các id đó và chỉ xử lý phần còn lại.

    HDF5 structure:
        /391895   → float16 array (257, 1024)
        /262148   → float16 array (257, 1024)
        ...

    Args:
        image_ids:   Danh sách unique image_id (int).
        img_dir:     Thư mục chứa ảnh .jpg.
        img_prefix:  Tiền tố tên file ảnh (e.g. "COCO_train2014_").
        cache_path:  Đường dẫn file HDF5 đầu ra (tạo mới hoặc append).
        processor:   CLIPImageProcessor.
        model:       CLIPVisionModel (eval, frozen).
        device:      "cuda" hoặc "cpu".
        batch_size:  Số ảnh mỗi batch GPU.
    """
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(str(cache_path), "a") as h5f:
        #  Resume: bỏ qua image_id đã cache 
        existing   = set(h5f.keys())
        to_process = [iid for iid in image_ids if str(iid) not in existing]
        print(f"Tổng image_id : {len(image_ids):,}")
        print(f"Đã cache      : {len(existing):,}")
        print(f"Còn lại       : {len(to_process):,}")

        if not to_process:
            print("Tất cả đã được cache. Bỏ qua.")
            return

        n_batches = (len(to_process) + batch_size - 1) // batch_size
        for i in tqdm(range(0, len(to_process), batch_size),
                      desc="Extracting", total=n_batches, unit="batch"):
            batch_ids = to_process[i : i + batch_size]
            imgs, valid_ids = [], []

            for iid in batch_ids:
                img_path = img_dir / f"{img_prefix}{iid:012d}.jpg"
                if not img_path.exists():
                    continue
                try:
                    imgs.append(Image.open(img_path).convert("RGB"))
                    valid_ids.append(iid)
                except Exception:
                    continue  # bỏ qua ảnh bị lỗi

            if not imgs:
                continue

            # Forward pass → [B, 257, 1024] float16
            feats = extract_batch_features(imgs, processor, model, device)

            # Lưu từng ảnh vào HDF5 (key = str(image_id))
            for j, iid in enumerate(valid_ids):
                h5f.create_dataset(
                    str(iid),
                    data=feats[j],
                    compression="gzip",
                    compression_opts=1,   # level 1 = nhanh, vẫn tiết kiệm ~30%
                )

    print(f"\n Xong! Cache: {cache_path}")

print("Hàm extract_and_cache đã định nghĩa ✓")

Hàm extract_and_cache đã định nghĩa ✓


In [ ]:
##  Cell 7: DEMO MODE — tạo cache giả với ảnh tổng hợp 
# Chạy cell này nếu chưa có COCO dataset để hiểu pipeline.
# Nếu đã có COCO dataset thật, bỏ qua cell này và chạy Cell 8.

DEMO_N_IMAGES   = 20      # số ảnh giả để demo
DEMO_CACHE_PATH = CONFIG["cache_dir"] / "demo_features.h5"
DEMO_IMAGE_IDS  = list(range(100001, 100001 + DEMO_N_IMAGES))

print(f"=== DEMO MODE: tạo {DEMO_N_IMAGES} ảnh ngẫu nhiên ===\n")

rng = np.random.default_rng(0)
t_start = time.time()

with h5py.File(str(DEMO_CACHE_PATH), "w") as h5f:
    for i in tqdm(range(0, DEMO_N_IMAGES, CONFIG["batch_size"]),
                  desc="Demo extract", unit="batch"):
        batch_ids = DEMO_IMAGE_IDS[i : i + CONFIG["batch_size"]]

        # Tạo ảnh ngẫu nhiên thay cho ảnh COCO thật
        dummy_imgs = [
            Image.fromarray((rng.random((224, 224, 3)) * 255).astype(np.uint8))
            for _ in batch_ids
        ]

        feats = extract_batch_features(
            dummy_imgs, processor, vit_model, CONFIG["device"]
        )  # [B, 257, 1024] float16

        for j, iid in enumerate(batch_ids):
            h5f.create_dataset(
                str(iid), data=feats[j],
                compression="gzip", compression_opts=1,
            )

elapsed = time.time() - t_start
print(f"\n Demo cache tạo xong  ({elapsed:.2f}s)")
print(f"Thời gian/ảnh : {elapsed/DEMO_N_IMAGES*1000:.1f} ms")
print(f"File           : {DEMO_CACHE_PATH}")
print(f"Dung lượng file: {DEMO_CACHE_PATH.stat().st_size / 1024:.1f} KB")

=== DEMO MODE: tạo 20 ảnh ngẫu nhiên ===



Demo extract:   0%|          | 0/1 [00:00<?, ?batch/s]


 Demo cache tạo xong  (63.87s)
Thời gian/ảnh : 3193.4 ms
File           : /content/drive/MyDrive/kì 2 năm 4/deep learning/blip2_project/data/cache/demo_features.h5
Dung lượng file: 9613.5 KB


In [ ]:
##  Cell 8: PRODUCTION — chạy extraction thật với COCO dataset (checkpoint per batch)
# Ghi cache sau mỗi batch + lưu checkpoint JSON → resume an toàn nếu bị gián đoạn.

import json as _json

# ── Tuning ──────────────────────────────────────────────────────────────────
CHECKPOINT_INTERVAL = 10   # flush HDF5 + lưu .ckpt.json sau mỗi N batch GPU
# ────────────────────────────────────────────────────────────────────────────


def extract_and_cache_with_checkpoint(
    image_ids: list,
    img_dir: Path,
    img_prefix: str,
    cache_path: Path,
    processor,
    model,
    device: str,
    batch_size: int = 32,
    checkpoint_interval: int = 10,
) -> None:
    """Giống extract_and_cache (Cell 6) nhưng bền hơn:

    1. Flush HDF5 sau mỗi `checkpoint_interval` batch GPU  → dữ liệu xuống disk
    2. Lưu checkpoint JSON song song (.ckpt.json)          → resume O(1) thay vì scan HDF5
    3. Khi khởi động lại: đọc checkpoint trước, merge với HDF5 keys để chắc chắn

    Checkpoint file: <cache_path>.ckpt.json
        { "done": [image_id, image_id, ...] }
    """
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    ckpt_path  = cache_path.with_suffix(".ckpt.json")   # e.g. train_features.ckpt.json

    # ── Bước 1: Xác định image_id đã xử lý ──────────────────────────────────
    done_ids: set = set()

    if ckpt_path.exists():
        with open(ckpt_path) as f:
            done_ids = set(_json.load(f).get("done", []))
        print(f"Checkpoint tìm thấy  : {ckpt_path.name}")
        print(f"  → {len(done_ids):,} ảnh đã ghi nhận trong checkpoint")

    # Merge với HDF5 keys (an toàn khi checkpoint bị mất nhưng HDF5 còn)
    if cache_path.exists():
        with h5py.File(str(cache_path), "r") as h5f:
            h5_done = {int(k) for k in h5f.keys()}
        newly_found = h5_done - done_ids
        if newly_found:
            print(f"  → Tìm thêm {len(newly_found):,} ảnh trong HDF5 (ngoài checkpoint)")
        done_ids = done_ids | h5_done

    to_process = [iid for iid in image_ids if iid not in done_ids]

    print(f"\nTổng image_id : {len(image_ids):,}")
    print(f"Đã cache      : {len(done_ids):,}")
    print(f"Còn lại       : {len(to_process):,}")

    if not to_process:
        print("✓ Tất cả đã được cache. Bỏ qua.")
        return

    # ── Bước 2: Extraction + checkpoint theo batch ────────────────────────────
    n_batches  = (len(to_process) + batch_size - 1) // batch_size
    saved_cnt  = 0   # số ảnh ghi thành công từ lúc bắt đầu run này

    with h5py.File(str(cache_path), "a") as h5f:
        pbar = tqdm(
            range(0, len(to_process), batch_size),
            total=n_batches, desc="Extracting", unit="batch",
        )

        for batch_num, i in enumerate(pbar):
            batch_ids = to_process[i : i + batch_size]
            imgs, valid_ids = [], []

            for iid in batch_ids:
                img_path = img_dir / f"{img_prefix}{iid:012d}.jpg"
                if not img_path.exists():
                    continue
                try:
                    imgs.append(Image.open(img_path).convert("RGB"))
                    valid_ids.append(iid)
                except Exception:
                    continue   # bỏ qua ảnh bị lỗi

            if not imgs:
                # batch toàn ảnh thiếu → đánh dấu done để không lặp lại
                done_ids.update(batch_ids)
                continue

            # Forward pass ViT → [B, 257, 1024] float16
            feats = extract_batch_features(imgs, processor, model, device)

            # Ghi vào HDF5 (skip nếu key đã tồn tại để tránh lỗi khi merge)
            for j, iid in enumerate(valid_ids):
                if str(iid) not in h5f:
                    h5f.create_dataset(
                        str(iid), data=feats[j],
                        compression="gzip", compression_opts=1,
                    )

            done_ids.update(valid_ids)
            saved_cnt += len(valid_ids)

            # ── Checkpoint sau mỗi checkpoint_interval batch ──────────────
            if (batch_num + 1) % checkpoint_interval == 0:
                h5f.flush()
                with open(ckpt_path, "w") as f:
                    _json.dump({"done": list(done_ids)}, f)
                pbar.set_postfix({
                    "saved_this_run": saved_cnt,
                    "total_cached": len(done_ids),
                    "ckpt": "✓",
                })

        # ── Flush + checkpoint cuối cùng (phần dư < checkpoint_interval) ──
        h5f.flush()
        with open(ckpt_path, "w") as f:
            _json.dump({"done": list(done_ids)}, f)

    print(f"\n✓ Xong!  Đã lưu {saved_cnt:,} ảnh trong run này.")
    print(f"  Cache      : {cache_path}")
    print(f"  Checkpoint : {ckpt_path}  ({len(done_ids):,} ảnh tổng)")


# ── Chạy production ──────────────────────────────────────────────────────────
if not (HAS_TRAIN_DATA or HAS_VAL_DATA):
    print("✗ Không có COCO dataset → bỏ qua cell này.")
    print("  Hãy đặt data_root đúng trong CONFIG (Cell 3) rồi chạy lại.")
else:
    for split, has_data, ann_path, img_dir, img_prefix, cache_path in [
        ("train", HAS_TRAIN_DATA,
         CONFIG["train_ann"], CONFIG["train_img_dir"],
         "COCO_train2014_", CONFIG["train_cache_h5"]),
        ("val",   HAS_VAL_DATA,
         CONFIG["val_ann"],   CONFIG["val_img_dir"],
         "COCO_val2014_",   CONFIG["val_cache_h5"]),
    ]:
        if not has_data:
            print(f"[{split}] Bỏ qua — không tìm thấy dữ liệu.")
            continue

        print(f"\n{'='*60}")
        print(f"Split: {split.upper()}")
        print(f"{'='*60}")

        with open(ann_path) as f:
            ann_data = _json.load(f)
        image_ids = sorted({ann["image_id"] for ann in ann_data["annotations"]})
        print(f"Unique image_id : {len(image_ids):,}")

        extract_and_cache_with_checkpoint(
            image_ids           = image_ids,
            img_dir             = img_dir,
            img_prefix          = img_prefix,
            cache_path          = cache_path,
            processor           = processor,
            model               = vit_model,
            device              = CONFIG["device"],
            batch_size          = CONFIG["batch_size"],
            checkpoint_interval = CHECKPOINT_INTERVAL,
        )



Split: TRAIN
Unique image_id : 82,783
Tổng image_id : 82,783
Đã cache      : 0
Còn lại       : 82,783


Extracting:   0%|          | 0/2587 [00:00<?, ?batch/s]

In [ ]:
##  Cell 9: Kiểm tra & thống kê file cache 
# Đọc file cache (demo hoặc thật) và in thông tin chi tiết

cache_to_inspect = DEMO_CACHE_PATH   # thay bằng CONFIG["train_cache_h5"] nếu có data thật

print(f"Kiểm tra file: {cache_to_inspect}")
print(f"Dung lượng   : {cache_to_inspect.stat().st_size / 1024:.1f} KB")
print()

with h5py.File(str(cache_to_inspect), "r") as h5f:
    keys      = list(h5f.keys())
    n_images  = len(keys)
    sample_id = keys[0]
    sample    = h5f[sample_id][:]   # load 1 sample

    print(f"{''*50}")
    print(f"Số image cached : {n_images:,}")
    print(f"Sample key      : '{sample_id}'  (= str(image_id))")
    print(f"Sample shape    : {sample.shape}")     # (257, 1024)
    print(f"Sample dtype    : {sample.dtype}")     # float16
    print(f" Ước tính dung lượng ")
    bytes_per_img = sample.nbytes                  # 257 × 1024 × 2 bytes
    print(f"  Mỗi ảnh (raw)  : {bytes_per_img/1024:.1f} KB  (float16)")
    print(f"  Mỗi ảnh (f32)  : {bytes_per_img*2/1024:.1f} KB  (float32 — gấp đôi)")
    n_train = 82_783
    print(f"  train2014 ({n_train:,} ảnh):")
    print(f"    float16 raw  : {bytes_per_img * n_train / 1e9:.2f} GB")
    print(f"    +gzip ~30%   : ~{bytes_per_img * n_train / 1e9 * 0.7:.2f} GB")
    print(f" Dataset so sánh ")
    coco_avg_jpg = 100 * 1024   # ~100 KB mỗi ảnh COCO JPEG
    print(f"  COCO raw JPEG  : ~{coco_avg_jpg * n_train / 1e9:.0f} GB  (ảnh gốc)")
    print(f"  Feature cache  : ~{bytes_per_img * n_train * 0.7 / 1e9:.2f} GB  (cache float16+gzip)")

    #  Kiểm tra shape của nhiều mẫu 
    print(f"\n Kiểm tra {min(5, n_images)} mẫu ")
    for k in keys[:5]:
        arr = h5f[k][:]
        status = "✓" if arr.shape == (CLIP_PATCH_TOKENS, CLIP_FEATURE_DIM) else "✗"
        print(f"  image_id={k:>8s}  shape={arr.shape}  dtype={arr.dtype}  {status}")

In [ ]:
##  Cell 10: Visualize features từ cache 
# Đọc 4 mẫu từ cache và visualize:
#  (a) Heatmap L2-norm của 256 patch tokens (reshape 16×16)
#  (b) Phân phối giá trị đặc trưng (histogram)

with h5py.File(str(cache_to_inspect), "r") as h5f:
    sample_keys    = list(h5f.keys())[:4]
    sample_feats   = [h5f[k][:].astype(np.float32) for k in sample_keys]
    # shape mỗi phần tử: (257, 1024)

PATCH_GRID = 16   # sqrt(256) = 16   ←  224/14 = 16

fig = plt.figure(figsize=(15, 8))
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.45, wspace=0.35)

for col, (k, feat) in enumerate(zip(sample_keys, sample_feats)):
    # feat: (257, 1024)
    cls_token   = feat[0]       # CLS token  [1024]
    patch_tokens= feat[1:]      # Patch tokens [256, 1024]

    #  Hàng trên: heatmap norm theo patch 
    patch_norms = np.linalg.norm(patch_tokens, axis=-1)   # [256]
    norm_map    = patch_norms.reshape(PATCH_GRID, PATCH_GRID)

    ax_top = fig.add_subplot(gs[0, col])
    im = ax_top.imshow(norm_map, cmap="plasma")
    ax_top.set_title(f"img_id={k}\nPatch norm ({PATCH_GRID}×{PATCH_GRID})", fontsize=9)
    ax_top.axis("off")
    plt.colorbar(im, ax=ax_top, fraction=0.046, pad=0.04)

    #  Hàng dưới: histogram giá trị đặc trưng 
    ax_bot = fig.add_subplot(gs[1, col])
    ax_bot.hist(patch_tokens.flatten(), bins=60, color="steelblue",
                alpha=0.75, edgecolor="none")
    ax_bot.axvline(cls_token.mean(), color="red", linestyle="--",
                   linewidth=1.2, label=f"CLS mean={cls_token.mean():.2f}")
    ax_bot.set_xlabel("Feature value", fontsize=8)
    ax_bot.set_ylabel("Count", fontsize=8)
    ax_bot.set_title("Phân phối patch features", fontsize=9)
    ax_bot.legend(fontsize=7)
    ax_bot.grid(True, alpha=0.3)

fig.suptitle(
    "Cell 10 — Visualize features đọc từ HDF5 cache\n"
    "Trên: L2-norm theo từng patch (16×16)    Dưới: Phân phối giá trị",
    fontsize=12, fontweight="bold"
)
plt.show()

#  In thống kê tổng hợp 
print(" Thống kê features ")
for k, feat in zip(sample_keys, sample_feats):
    p = feat[1:]
    print(f"  img={k}  patch norm  mean={np.linalg.norm(p, axis=-1).mean():.2f}  "
          f"feat mean={p.mean():.3f}  std={p.std():.3f}")

In [ ]:
##  Cell 11: Benchmark — tốc độ đọc cache vs forward pass 
# So sánh thực tế: đọc từ HDF5 vs chạy ViT forward pass lại

N_BENCHMARK = min(16, DEMO_N_IMAGES)

#  Cách 1: Đọc từ HDF5 cache 
t0 = time.time()
with h5py.File(str(cache_to_inspect), "r") as h5f:
    loaded_feats = []
    for k in list(h5f.keys())[:N_BENCHMARK]:
        feat = torch.from_numpy(h5f[k][:].astype(np.float32))
        loaded_feats.append(feat)
t_cache = time.time() - t0

#  Cách 2: Forward pass lại 
rng_bench   = np.random.default_rng(99)
dummy_batch = [
    Image.fromarray((rng_bench.random((224, 224, 3)) * 255).astype(np.uint8))
    for _ in range(N_BENCHMARK)
]

t0 = time.time()
_ = extract_batch_features(dummy_batch, processor, vit_model, CONFIG["device"])
t_vit = time.time() - t0

#  Kết quả 
print(f"{''*55}")
print(f"Benchmark: {N_BENCHMARK} ảnh")
print(f"{''*55}")
print(f"  Đọc HDF5 cache    : {t_cache*1000:7.2f} ms  "
      f"({t_cache/N_BENCHMARK*1000:.2f} ms/ảnh)")
print(f"  ViT forward pass  : {t_vit*1000:7.2f} ms  "
      f"({t_vit/N_BENCHMARK*1000:.2f} ms/ảnh)")
speedup = t_vit / t_cache if t_cache > 0 else float("inf")
print(f"{''*55}")
print(f"  Speedup cache / ViT : {speedup:.1f}×")
print()

#  Biểu đồ so sánh 
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(
    ["Đọc từ\nHDF5 cache", "ViT forward\npass"],
    [t_cache * 1000, t_vit * 1000],
    color=["#2ecc71", "#e74c3c"],
    width=0.4,
    edgecolor="white",
)
ax.bar_label(bars, fmt="%.1f ms", padding=4, fontsize=11, fontweight="bold")
ax.set_ylabel("Thời gian (ms)", fontsize=11)
ax.set_title(
    f"Benchmark: đọc cache vs ViT forward pass ({N_BENCHMARK} ảnh)\n"
    f"Cache nhanh hơn {speedup:.1f}× → {(t_vit-t_cache)*1000:.1f} ms tiết kiệm mỗi batch",
    fontsize=11
)
ax.set_ylim(0, max(t_cache, t_vit) * 1000 * 1.25)
ax.grid(True, axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
##  Cell 12: Cách DataLoader đọc từ cache (tích hợp với VQAv2Dataset) 
# Minh hoạ cách VQAv2Dataset sử dụng cache khi use_cache=True
# (dựa theo data/vqa_dataset.py của dự án)

class MiniCachedDataset(torch.utils.data.Dataset):
    """Dataset tối giản minh hoạ cách đọc features từ HDF5.

    Trong dự án thật, VQAv2Dataset.use_cache=True sẽ làm điều tương tự.
    """

    def __init__(self, image_ids: list, cache_path: Path):
        self.image_ids  = image_ids
        self.cache_path = str(cache_path)
        # HDF5 file được mở một lần và giữ trong suốt vòng đời dataset
        self._h5 = h5py.File(self.cache_path, "r")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx: int) -> dict:
        iid  = self.image_ids[idx]
        feat = self._h5[str(iid)][:]                     # (257, 1024) float16
        return {
            "image_features": torch.from_numpy(feat.astype(np.float32)),  # [257, 1024]
            "image_id"      : iid,
        }

    def __del__(self):
        if hasattr(self, "_h5") and self._h5.id.valid:
            self._h5.close()


#  Test DataLoader 
ds     = MiniCachedDataset(DEMO_IMAGE_IDS, DEMO_CACHE_PATH)
loader = torch.utils.data.DataLoader(ds, batch_size=4, shuffle=False, num_workers=0)

print(f"Dataset size : {len(ds)}")
print(f"Batch size   : 4")
print()

for batch_idx, batch in enumerate(loader):
    feat = batch["image_features"]   # [4, 257, 1024]
    iids = batch["image_id"]
    print(f"Batch {batch_idx}:")
    print(f"  image_features shape : {list(feat.shape)}")
    print(f"  image_features dtype : {feat.dtype}")
    print(f"  image_ids            : {iids.tolist()}")
    print()
    if batch_idx >= 2:
        print("  ... (hiển thị 3 batch đầu)")
        break

print(" Đây là dữ liệu đưa vào Q-Former ")
print(f"Q-Former input: [B=4, N=257, D=1024]  ← image_features")
print(f"                 B = batch size")
print(f"                 N = CLIP_PATCH_TOKENS = 257")
print(f"                 D = CLIP_FEATURE_DIM  = 1024")

In [ ]:
##  Cell 13: Tổng kết 

print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              TÓM TẮT: PRE-EXTRACT & CACHE IMAGE FEATURES                 ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  INPUT                                                                   ║
║  ├ COCO images    : COCO_{split}2014_000000{id:012d}.jpg               ║
║  ├ Annotations    : v2_mscoco_{split}2014_annotations.json             ║
║  └ Model          : openai/clip-vit-large-patch14  (frozen)            ║
║                                                                          ║
║  XỬ LÝ                                                                   ║
║  1. Đọc image_id từ annotations → dedup (unique)                         ║
║  2. Batch ảnh → CLIPImageProcessor (resize 224, normalize CLIP)          ║
║  3. ViT-L/14 forward → last_hidden_state [B, 257, 1024]                  ║
║  4. float32 → float16  (tiết kiệm 50% RAM/disk)                          ║
║  5. Lưu vào HDF5 (key=str(image_id), compression=gzip lv1)               ║
║  6. Hỗ trợ RESUME: bỏ qua id đã cache                                    ║
║                                                                          ║
║  OUTPUT                                                                  ║
║  ├ train_features.h5   (82,783 keys × float16[257,1024])               ║
║  └ val_features.h5     (40,504 keys × float16[257,1024])               ║
║                                                                          ║
║  LỢI ÍCH                                                                 ║
║  ├ Training không cần chạy ViT nữa → nhanh 3-5×                        ║
║  ├ Giải phóng GPU cho Q-Former & LM                                    ║
║  └ Cache dùng lại nhiều lần, nhiều thí nghiệm                          ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")